Task 1 si 2

In [12]:
import nltk

# ---------- TASK 1: gramatica CFG ----------
grammar = nltk.CFG.fromstring("""
S       -> NP VP

NP      -> Det Nominal | Nominal | VPger | NP PP
Nominal -> Noun | Adj Nominal | Nominal Conj Nominal
VPger   -> Ger NP

VP      -> V NP | V Adv | Aux VP | Modal VP | Cop Adj | VP PP
PP      -> Prep NP

Det     -> 'the'
Noun    -> 'professors' | 'men' | 'women' | 'reporter' | 'actor' | 'camera' | 'entrance'
Adj     -> 'visiting' | 'old' | 'annoying'
Ger     -> 'visiting'
Modal   -> 'can'
Cop     -> 'be'
Aux     -> 'were'
V       -> 'waiting' | 'interviewed'
Adv     -> 'outside'
Conj    -> 'and'
Prep    -> 'with' | 'near'
""")

sentences = [
    "Visiting professors can be annoying.",
    "Old men and women were waiting outside.",
    "The reporter interviewed the actor with the camera near the entrance.",
]

# ---------- TASK 2: arbori de structura sintagmatica ----------
def tokenize(s):
    return s.lower().rstrip(".").split()

for parser_cls in (nltk.ChartParser, nltk.EarleyChartParser):
    parser = parser_cls(grammar)
    print("=" * 70)
    print(parser_cls.__name__)
    for s in sentences:
        trees = list(parser.parse(tokenize(s)))
        print(f"\n{s}  ->  {len(trees)} arbori")
        for i, t in enumerate(trees, 1):
            print(f"--- arbore {i}")
            print(t)

ChartParser

Visiting professors can be annoying.  ->  2 arbori
--- arbore 1
(S
  (NP (Nominal (Adj visiting) (Nominal (Noun professors))))
  (VP (Modal can) (VP (Cop be) (Adj annoying))))
--- arbore 2
(S
  (NP (VPger (Ger visiting) (NP (Nominal (Noun professors)))))
  (VP (Modal can) (VP (Cop be) (Adj annoying))))

Old men and women were waiting outside.  ->  2 arbori
--- arbore 1
(S
  (NP
    (Nominal
      (Nominal (Adj old) (Nominal (Noun men)))
      (Conj and)
      (Nominal (Noun women))))
  (VP (Aux were) (VP (V waiting) (Adv outside))))
--- arbore 2
(S
  (NP
    (Nominal
      (Adj old)
      (Nominal
        (Nominal (Noun men))
        (Conj and)
        (Nominal (Noun women)))))
  (VP (Aux were) (VP (V waiting) (Adv outside))))

The reporter interviewed the actor with the camera near the entrance.  ->  5 arbori
--- arbore 1
(S
  (NP (Det the) (Nominal (Noun reporter)))
  (VP
    (V interviewed)
    (NP
      (NP
        (NP (Det the) (Nominal (Noun actor)))
        (PP (Pre

Task 3. (0.25p) Implement a dependency parser (using, for instance, spaCY, NLTK or Stanza) and
parse the three sentences above.

In [11]:
import spacy
from spacy import displacy


nlp = spacy.load("en_core_web_sm")

sentences = [
    "Visiting professors can be annoying.",
    "Old men and women were waiting outside.",
    "The reporter interviewed the actor with the camera near the entrance."
]

for i, sent_text in enumerate(sentences, 1):
    print(f"\n--- Propoziția {i}: \"{sent_text}\" ---")
    doc = nlp(sent_text)

    #(Head -> Dependent [Label])
    print(f"{'Cuvânt (Dependent)':<15} | {'Relație (DEPREL)':<15} | {'Head (Cuvântul Guvernator)'}")
    print("-" * 55)
    for token in doc:
        print(f"{token.text:<15} | {token.dep_:<15} | {token.head.text}")




--- Propoziția 1: "Visiting professors can be annoying." ---
Cuvânt (Dependent) | Relație (DEPREL) | Head (Cuvântul Guvernator)
-------------------------------------------------------
Visiting        | amod            | professors
professors      | nsubj           | be
can             | aux             | be
be              | ROOT            | be
annoying        | acomp           | be
.               | punct           | be

--- Propoziția 2: "Old men and women were waiting outside." ---
Cuvânt (Dependent) | Relație (DEPREL) | Head (Cuvântul Guvernator)
-------------------------------------------------------
Old             | amod            | men
men             | nsubj           | waiting
and             | cc              | men
women           | conj            | men
were            | aux             | waiting
waiting         | ROOT            | waiting
outside         | advmod          | waiting
.               | punct           | waiting

--- Propoziția 3: "The reporter interviewed 

Task 4. (0,30p) Write a program which transforms a Context Free Grammar, as the one discussed in
class,
into a Chomsky Normal Form grammar, also converting the probabilities. Test it on the
grammar from the course.

In [10]:
import nltk
from nltk import PCFG
from nltk.grammar import Nonterminal, ProbabilisticProduction
from collections import defaultdict

course_pcfg = PCFG.fromstring("""
    S -> NP VP [0.8]
    S -> Aux NP VP [0.1]
    S -> VP [0.1]

    NP -> Pronoun [0.2]
    NP -> Proper_Noun [0.2]
    NP -> Det Nominal [0.6]

    Nominal -> Noun [0.3]
    Nominal -> Nominal Noun [0.2]
    Nominal -> Nominal PP [0.5]

    VP -> Verb [0.2]
    VP -> Verb NP [0.5]
    VP -> VP PP [0.3]

    PP -> Prep NP [1.0]

    Det -> 'the' [0.6] | 'a' [0.2] | 'that' [0.1] | 'this' [0.1]
    Noun -> 'book' [0.3] | 'flight' [0.3] | 'meal' [0.2] | 'money' [0.2]
    Pronoun -> 'I' [0.25] | 'he' [0.25] | 'she' [0.25] | 'me' [0.25]
    Proper_Noun -> 'Houston' [0.8] | 'NWA' [0.2]
    Verb -> 'book' [0.3] | 'include' [0.4] | 'prefer' [0.3]
    Aux -> 'does' [1.0]
    Prep -> 'from' [0.25] | 'to' [0.25] | 'on' [0.1] | 'near' [0.2] | 'through' [0.2]
""")
# NB: am folosit Proper_Noun (cu underscore), pentru că '-' poate crea probleme la parsare.


def convert_pcfg_to_cnf(pcfg):
    counter = 1
    prods = list(pcfg.productions())

    # --- Pasul 1: terminalele din reguli cu RHS > 1 devin nonterminali (TERM) ---
    step1 = []
    term_nts = {}
    for p in prods:
        rhs = p.rhs()
        if len(rhs) > 1:
            new_rhs = []
            for sym in rhs:
                if not isinstance(sym, Nonterminal):
                    if sym not in term_nts:
                        term_nts[sym] = Nonterminal(f"T_{str(sym).upper()}")
                    new_rhs.append(term_nts[sym])
                else:
                    new_rhs.append(sym)
            step1.append(ProbabilisticProduction(p.lhs(), new_rhs, prob=p.prob()))
        else:
            step1.append(p)
    for t, nt in term_nts.items():
        step1.append(ProbabilisticProduction(nt, [t], prob=1.0))

    # --- Pasul 2: BIN - reguli cu RHS > 2 ---
    step2 = []
    for p in step1:
        rhs = p.rhs()
        if len(rhs) > 2:
            current_lhs = p.lhs()
            prob = p.prob()
            for i in range(len(rhs) - 2):
                new_nt = Nonterminal(f"X{counter}")
                counter += 1
                step2.append(ProbabilisticProduction(current_lhs, [rhs[i], new_nt], prob=prob))
                current_lhs = new_nt
                prob = 1.0  # doar prima regulă poartă probabilitatea originală
            step2.append(ProbabilisticProduction(current_lhs, [rhs[-2], rhs[-1]], prob=1.0))
        else:
            step2.append(p)

    # --- Pasul 3: UNIT - eliminarea regulilor A -> B (B nonterminal) ---
    unit = defaultdict(dict)   # unit[A][B] = prob
    non_unit = []
    for p in step2:
        if len(p.rhs()) == 1 and isinstance(p.rhs()[0], Nonterminal):
            unit[p.lhs()][p.rhs()[0]] = p.prob()
        else:
            non_unit.append(p)

    # Închidere tranzitivă: prob(A =>* B) ca produs de probabilități (maxim peste căi)
    def closure(A):
        result = {A: 1.0}
        stack = [A]
        while stack:
            X = stack.pop()
            for Y, pr in unit.get(X, {}).items():
                new_p = result[X] * pr
                if new_p > result.get(Y, 0.0):
                    result[Y] = new_p
                    stack.append(Y)
        return result

    by_lhs = defaultdict(list)
    for p in non_unit:
        by_lhs[p.lhs()].append(p)

    all_lhs = set(p.lhs() for p in step2)
    final = {}
    for A in all_lhs:
        for B, pr_AB in closure(A).items():
            for p in by_lhs.get(B, []):
                key = (A, p.rhs())
                final[key] = final.get(key, 0.0) + pr_AB * p.prob()

    final_prods = [ProbabilisticProduction(lhs, list(rhs), prob=pr)
                   for (lhs, rhs), pr in final.items()]
    return PCFG(pcfg.start(), final_prods)


cnf_pcfg = convert_pcfg_to_cnf(course_pcfg)

print("=== GRAMATICA CONVERTITĂ ÎN CNF ===")
for production in cnf_pcfg.productions():
    print(production)

=== GRAMATICA CONVERTITĂ ÎN CNF ===
PP -> Prep NP [1.0]
X1 -> NP VP [1.0]
Nominal -> Nominal Noun [0.2]
Nominal -> Nominal PP [0.5]
Nominal -> 'book' [0.09]
Nominal -> 'flight' [0.09]
Nominal -> 'meal' [0.06]
Nominal -> 'money' [0.06]
Noun -> 'book' [0.3]
Noun -> 'flight' [0.3]
Noun -> 'meal' [0.2]
Noun -> 'money' [0.2]
NP -> Det Nominal [0.6]
NP -> 'I' [0.05]
NP -> 'he' [0.05]
NP -> 'she' [0.05]
NP -> 'me' [0.05]
NP -> 'Houston' [0.16]
NP -> 'NWA' [0.04]
Aux -> 'does' [1.0]
Prep -> 'from' [0.25]
Prep -> 'to' [0.25]
Prep -> 'on' [0.1]
Prep -> 'near' [0.2]
Prep -> 'through' [0.2]
Det -> 'the' [0.6]
Det -> 'a' [0.2]
Det -> 'that' [0.1]
Det -> 'this' [0.1]
S -> NP VP [0.8]
S -> Aux X1 [0.1]
S -> Verb NP [0.05]
S -> VP PP [0.03]
S -> 'book' [0.006]
S -> 'include' [0.008]
S -> 'prefer' [0.006]
VP -> Verb NP [0.5]
VP -> VP PP [0.3]
VP -> 'book' [0.06]
VP -> 'include' [0.08]
VP -> 'prefer' [0.06]
Proper_Noun -> 'Houston' [0.8]
Proper_Noun -> 'NWA' [0.2]
Verb -> 'book' [0.3]
Verb -> 'include' 

5. (0,15p) Describe an application n which constituency and/or dependency parsing can provide
useful information. Explain briefly why. (1/2 pag.)

O aplicație în care analiza sintactică aduce informații utile este extragerea de relații din text, de exemplu construirea automată a unei baze de cunoștințe din articole de știri, pornind de la propoziții precum “The reporter interviewed the actor with the camera near the entrance.”

Rolul analizei de dependențe. Un parser de dependențe identifică direct relațiile gramaticale dintre cuvinte: subiectul (nsubj), obiectul direct (dobj) și modificatorii. Pentru exemplul de mai sus obținem triplete de forma (reporter, interviewed, actor), adică cine a făcut ce și cui. Aceste triplete se pot extrage independent de ordinea cuvintelor sau de lungimea propoziției, ceea ce o simplă căutare după cuvinte-cheie sau după poziție nu poate face. Tot așa, în propoziții pasive sau cu incidente lungi, subiectul logic rămâne ușor de recuperat din arborele de dependențe.

Rolul analizei de constituenți. Arborele de structură sintagmatică delimitează grupurile nominale și verbale (NP, VP, PP), deci permite extragerea entităților complete (“the actor with the camera”) și nu doar a cuvintelor izolate. În plus, structura arborelui arată ambiguitățile: în propoziția dată, sintagma prepozițională “with the camera” poate atașa fie la verb (reporterul a intervievat actorul folosind camera), fie la substantivul actor (actorul care avea camera). Cele două arbori corespund celor două sensuri, iar un sistem care le poate distinge poate alege interpretarea corectă sau poate semnala ambiguitatea. La fel, “Visiting professors can be annoying” are două analize (profesorii care vizitează sunt enervanți / a vizita profesori este enervant), iar “Old men and women” poate însemna (old men) and women sau old (men and women).

De ce este util. Fără informații sintactice, un sistem bazat doar pe cuvinte (bag-of-words) nu poate deosebi “the reporter interviewed the actor” de “the actor interviewed the reporter”, deși conțin aceleași cuvinte. Analiza sintactică oferă tocmai această structură, de care depind aplicații precum răspunsul la întrebări, traducerea automată (reordonarea cuvintelor între limbi), analiza de sentiment (de exemplu, aria de acțiune a negației) și rezumarea automată. În practică, analiza de dependențe este preferată în extragerea de relații, pentru că este mai compactă și mai ușor de folosit pe date reale, iar analiza de constituenți este utilă atunci când contează delimitarea completă a sintagmelor și tratarea ambiguităților de atașare.